# RescueVision Edge — YOLOv5n Baseline Experiment

**Purpose:** Establish YOLOv5n baseline metrics on VisDrone pedestrian detection.  
This notebook documents the empirical justification for architecture selection.

**Result of this notebook feeds directly into Proposal Bab 2 (Model Selection Rationale).**

---
Hypothesis: YOLOv8n outperforms YOLOv5n on small-object detection (VisDrone pedestrian)  
due to anchor-free head and improved C2f backbone, with comparable CPU inference time.

**If hypothesis is FALSE → revert to YOLOv5n as final model.**  
**If hypothesis is TRUE → YOLOv8n is justified as final model.**

## 0. Environment

In [1]:
import os
import sys
import time
import numpy as np
import torch
from pathlib import Path
from ultralytics import YOLO
import ultralytics

print(f"Ultralytics: {ultralytics.__version__}")

REPO_ROOT = Path(os.getcwd()).parent
os.chdir(REPO_ROOT)

print(f'Repo root: {REPO_ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

device = 'cuda' if torch.cuda.is_available() else \
         'mps' if (hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()) else 'cpu'
print(f'Training device: {device}')

# Verify dataset
for split in ['train_data/images/train', 'train_data/images/val']:
    p = REPO_ROOT / split
    n = len(list(p.glob('*.jpg')) + list(p.glob('*.png'))) if p.exists() else 0
    print(f'{split}: {n} images')

Ultralytics: 8.4.31
Repo root: c:\Users\MASTER CORE\RescueVision
PyTorch: 2.5.1+cu121
CUDA available: True
Training device: cuda
train_data/images/train: 5655 images
train_data/images/val: 530 images


## 1. Install YOLOv5 (Ultralytics repo)

In [2]:
# YOLOv5 uses a different repo than YOLOv8 (not the ultralytics package)
# Clone if not already present
yolov5_dir = REPO_ROOT / 'yolov5'

if not yolov5_dir.exists():
    model = YOLO('yolov5n.pt')  # Auto-download jika belum ada
    print(model.info())
else:
    print(f'YOLOv5 already at: {yolov5_dir}')

# Add to path
sys.path.insert(0, str(yolov5_dir))
print(f'YOLOv5 path added.')

YOLOv5 already at: c:\Users\MASTER CORE\RescueVision\yolov5
YOLOv5 path added.


## 2. Training Configuration

In [3]:
# ============================================================
# YOLOV5N CONFIG — kept identical to YOLOv8n config where possible
# for fair comparison
# ============================================================

BASELINE_CONFIG = {
    'model'   : 'yolov5n.pt',       # YOLOv5 nano
    'data'    : str(REPO_ROOT / 'dataset.yaml'),
    'epochs'  : 100,
    'patience': 20,                  # Early stopping
    'batch'   : 16,
    'imgsz'   : 640,                 # SAME as YOLOv8n for fair comparison
    'device'  : device,
    'project' : str(REPO_ROOT / 'runs'),
    'name'    : 'yolov5n_baseline',
    'exist_ok': True,
}

print('YOLOv5n Baseline Config:')
for k, v in BASELINE_CONFIG.items():
    print(f'  {k:12s}: {v}')

print()
print('NOTE: imgsz=640 is intentionally identical to YOLOv8n experiment.')
print('Changing imgsz would invalidate the comparison.')

YOLOv5n Baseline Config:
  model       : yolov5n.pt
  data        : c:\Users\MASTER CORE\RescueVision\dataset.yaml
  epochs      : 100
  patience    : 20
  batch       : 16
  imgsz       : 640
  device      : cuda
  project     : c:\Users\MASTER CORE\RescueVision\runs
  name        : yolov5n_baseline
  exist_ok    : True

NOTE: imgsz=640 is intentionally identical to YOLOv8n experiment.
Changing imgsz would invalidate the comparison.


## 3. Train YOLOv5n

In [5]:
import subprocess

train_cmd = [
    sys.executable, str(yolov5_dir / 'train.py'),
    '--img',      str(BASELINE_CONFIG['imgsz']),
    '--batch',    str(BASELINE_CONFIG['batch']),
    '--epochs',   str(BASELINE_CONFIG['epochs']),
    '--data',     BASELINE_CONFIG['data'],
    '--weights',  BASELINE_CONFIG['model'],
    '--device',   '0' if device == 'cuda' else device,
    '--project',  BASELINE_CONFIG['project'],
    '--name',     BASELINE_CONFIG['name'],
    '--patience', str(BASELINE_CONFIG['patience']),
    '--exist-ok',
]

print('Running: ' + ' '.join(train_cmd))
print()

t0 = time.time()
result = subprocess.run(train_cmd, capture_output=False, text=True)
train_time = time.time() - t0

print(f'\nTraining exit code: {result.returncode}')
print(f'Training time: {train_time/3600:.2f} hours')

Running: c:\Users\MASTER CORE\miniconda3\envs\mlenv\python.exe c:\Users\MASTER CORE\RescueVision\yolov5\train.py --img 640 --batch 16 --epochs 100 --data c:\Users\MASTER CORE\RescueVision\dataset.yaml --weights yolov5n.pt --device 0 --project c:\Users\MASTER CORE\RescueVision\runs --name yolov5n_baseline --patience 20 --exist-ok


Training exit code: 0
Training time: 1.15 hours


## 4. Evaluate YOLOv5n

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

run_dir = REPO_ROOT / 'runs' / BASELINE_CONFIG['name']
best_weights = run_dir / 'weights' / 'best.pt'

print(f'Run dir: {run_dir}')
print(f'Best weights: {best_weights} (exists: {best_weights.exists()})')

results_csv = run_dir / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    final = df.iloc[-1]
    
    # YOLOv5 column names differ slightly from YOLOv8
    possible_map50_cols = ['metrics/mAP_0.5', 'metrics/mAP50(B)', 'mAP_0.5']
    possible_map_cols   = ['metrics/mAP_0.5:0.95', 'metrics/mAP50-95(B)', 'mAP_0.5:0.95']
    
    map50    = next((final[c] for c in possible_map50_cols if c in df.columns), None)
    map5095  = next((final[c] for c in possible_map_cols   if c in df.columns), None)
    
    print(f'\nYOLOv5n Final Metrics (val set):')
    print(f'  mAP@0.5:      {map50:.4f}' if map50 else '  mAP@0.5:      N/A')
    print(f'  mAP@0.5:0.95: {map5095:.4f}' if map5095 else '  mAP@0.5:0.95: N/A')
    print(f'  Available cols: {list(df.columns)}')
else:
    print('results.csv not found — training may have failed')
    map50, map5095 = None, None

Run dir: c:\Users\MASTER CORE\RescueVision\runs\yolov5n_baseline
Best weights: c:\Users\MASTER CORE\RescueVision\runs\yolov5n_baseline\weights\best.pt (exists: True)

YOLOv5n Final Metrics (val set):
  mAP@0.5:      0.4679
  mAP@0.5:0.95: 0.1728
  Available cols: ['epoch', 'train/box_loss', 'train/obj_loss', 'train/cls_loss', 'metrics/precision', 'metrics/recall', 'metrics/mAP_0.5', 'metrics/mAP_0.5:0.95', 'val/box_loss', 'val/obj_loss', 'val/cls_loss', 'x/lr0', 'x/lr1', 'x/lr2']


## 5. Export YOLOv5n to ONNX & CPU Benchmark

In [7]:
if best_weights.exists():
    # Export to ONNX
    export_cmd = [
        sys.executable, str(yolov5_dir / 'export.py'),
        '--weights', str(best_weights),
        '--include', 'onnx',
        '--imgsz', str(BASELINE_CONFIG['imgsz']),
        '--opset', '12',
        '--simplify',
        '--device', 'cpu',
    ]
    subprocess.run(export_cmd, capture_output=False, text=True)
    
    # Check ONNX output
    onnx_path = best_weights.with_suffix('.onnx')
    if onnx_path.exists():
        onnx_size_mb = os.path.getsize(onnx_path) / 1e6
        print(f'YOLOv5n ONNX: {onnx_size_mb:.2f} MB')
    else:
        print('ONNX export failed — check logs above')
        onnx_size_mb = None
else:
    print('No weights found — skip export')
    onnx_size_mb = None
    onnx_path = None

YOLOv5n ONNX: 7.49 MB


In [9]:
# CPU inference benchmark for YOLOv5n
import onnxruntime as ort
import cv2

def benchmark_onnx_cpu(onnx_path, images_dir, n=20, input_size=416):
    """Benchmark ONNX model on CPU. Returns timing stats in ms."""
    session = ort.InferenceSession(
        str(onnx_path),
        providers=['CPUExecutionProvider']
    )
    input_name = session.get_inputs()[0].name
    
    image_paths = sorted(
        list(Path(images_dir).glob('*.jpg')) + list(Path(images_dir).glob('*.png'))
    )[:n]
    
    if not image_paths:
        print(f'No images in {images_dir}')
        return None
    
    # Warmup
    dummy = np.random.rand(1, 3, input_size, input_size).astype(np.float32)
    for _ in range(3):
        session.run(None, {input_name: dummy})
    
    # Benchmark
    times = []
    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        img = cv2.resize(img, (input_size, input_size))
        tensor = img.astype(np.float32) / 255.0
        tensor = np.transpose(tensor, (2,0,1))[np.newaxis]
        
        t0 = time.perf_counter()
        session.run(None, {input_name: tensor})
        times.append((time.perf_counter() - t0) * 1000)
    
    return np.array(times)

if onnx_path and onnx_path.exists():
    print('Benchmarking YOLOv5n on CPU...')
    times_v5 = benchmark_onnx_cpu(
        onnx_path,
        REPO_ROOT / 'test_data' / 'images',
        n=20,
        input_size=640
    )
    if times_v5 is not None:
        print(f'YOLOv5n CPU timing (20 samples):')
        print(f'  Mean: {times_v5.mean():.1f} ms')
        print(f'  P95:  {np.percentile(times_v5, 95):.1f} ms')
        print(f'  Max:  {times_v5.max():.1f} ms')
        print(f'  Constraint C-A3: {"PASS" if times_v5.max() < 3000 else "FAIL"}')
else:
    times_v5 = None
    print('Skipping benchmark — no ONNX model')

Benchmarking YOLOv5n on CPU...
YOLOv5n CPU timing (20 samples):
  Mean: 18.0 ms
  P95:  19.7 ms
  Max:  19.8 ms
  Constraint C-A3: PASS


## 6. Architecture Comparison: YOLOv5n vs YOLOv8n

**Fill in the YOLOv8n results from `training.ipynb` after running both.**

In [13]:
# ============================================================
# FILL THESE IN after running training.ipynb (YOLOv8n)
# ============================================================
YOLOV8N_MAP50       = 0.3038   # e.g. 0.312  ← from training.ipynb cell 4
YOLOV8N_MAP5095     = 0.1157   # e.g. 0.198
YOLOV8N_SIZE_MB     = 11.70   # e.g. 11.4   ← model.onnx size
YOLOV8N_CPU_MEAN_MS = 30.4   # e.g. 280    ← from benchmark_cpu.py
YOLOV8N_CPU_MAX_MS  = 33.7   # e.g. 420

# Auto-read YOLOv5n results
v5_map50    = float(map50)    if map50    else None
v5_map5095  = float(map5095)  if map5095  else None
v5_size_mb  = onnx_size_mb
v5_mean_ms  = float(times_v5.mean()) if times_v5 is not None else None
v5_max_ms   = float(times_v5.max())  if times_v5 is not None else None

# Print comparison table
print('='*65)
print(f'{"Metric":<28} {"YOLOv5n":>15} {"YOLOv8n":>15}')
print('='*65)

rows = [
    ('mAP@0.5 (val)',        v5_map50,    YOLOV8N_MAP50,       '{:.4f}'),
    ('mAP@0.5:0.95 (val)',   v5_map5095,  YOLOV8N_MAP5095,     '{:.4f}'),
    ('ONNX size (MB)',        v5_size_mb,  YOLOV8N_SIZE_MB,     '{:.2f}'),
    ('CPU mean latency (ms)', v5_mean_ms,  YOLOV8N_CPU_MEAN_MS, '{:.1f}'),
    ('CPU max latency (ms)',  v5_max_ms,   YOLOV8N_CPU_MAX_MS,  '{:.1f}'),
]

for label, v5_val, v8_val, fmt in rows:
    v5_str = fmt.format(v5_val) if v5_val is not None else 'pending'
    v8_str = fmt.format(v8_val) if v8_val is not None else 'pending'
    print(f'{label:<28} {v5_str:>15} {v8_str:>15}')

print('='*65)
print()
print('Constraint C-A1 (≤50MB):')
for name, val in [('YOLOv5n', v5_size_mb), ('YOLOv8n', YOLOV8N_SIZE_MB)]:
    if val: print(f'  {name}: {"PASS" if val < 50 else "FAIL"} ({val:.2f} MB)')
print('Constraint C-A3 (≤3000ms):')
for name, val in [('YOLOv5n', v5_max_ms), ('YOLOv8n', YOLOV8N_CPU_MAX_MS)]:
    if val: print(f'  {name}: {"PASS" if val < 3000 else "FAIL"} ({val:.1f} ms)')

Metric                               YOLOv5n         YOLOv8n
mAP@0.5 (val)                         0.4679          0.3038
mAP@0.5:0.95 (val)                    0.1728          0.1157
ONNX size (MB)                          7.49           11.70
CPU mean latency (ms)                   18.0            30.4
CPU max latency (ms)                    19.8            33.7

Constraint C-A1 (≤50MB):
  YOLOv5n: PASS (7.49 MB)
  YOLOv8n: PASS (11.70 MB)
Constraint C-A3 (≤3000ms):
  YOLOv5n: PASS (19.8 ms)
  YOLOv8n: PASS (33.7 ms)


In [14]:
# ============================================================
# DECISION CELL — Architecture Selection
# Run this LAST, after both models are trained and benchmarked
# ============================================================

all_filled = all(x is not None for x in [
    YOLOV8N_MAP50, YOLOV8N_MAP5095, YOLOV8N_SIZE_MB, YOLOV8N_CPU_MEAN_MS, YOLOV8N_CPU_MAX_MS,
    v5_map50, v5_size_mb, v5_max_ms
])

if not all_filled:
    print('Waiting for both models to complete. Fill in YOLOV8N_* variables above.')
else:
    v8_wins_accuracy = YOLOV8N_MAP50 > v5_map50
    v8_within_speed  = YOLOV8N_CPU_MAX_MS < 3000  # Both must pass constraint
    v8_within_size   = YOLOV8N_SIZE_MB < 50
    
    delta_map50 = YOLOV8N_MAP50 - v5_map50
    delta_speed = YOLOV8N_CPU_MEAN_MS - v5_mean_ms
    
    print('ARCHITECTURE SELECTION DECISION')
    print('='*50)
    print(f'YOLOv8n accuracy improvement: {delta_map50:+.4f} mAP@0.5')
    print(f'YOLOv8n speed delta:          {delta_speed:+.1f} ms (mean)')
    print(f'YOLOv8n passes all constraints: {v8_within_speed and v8_within_size}')
    print()
    
    if v8_wins_accuracy and v8_within_speed and v8_within_size:
        decision = 'YOLOv8n'
        reason = (
            f'YOLOv8n achieves {delta_map50:+.4f} higher mAP@0.5 than YOLOv5n '
            f'while satisfying all constraints (size: {YOLOV8N_SIZE_MB:.2f}MB, '
            f'max latency: {YOLOV8N_CPU_MAX_MS:.0f}ms). '
            f'The anchor-free detection head provides better localization '
            f'for small objects typical of aerial imagery.'
        )
    elif not v8_within_speed or not v8_within_size:
        decision = 'YOLOv5n'
        reason = 'YOLOv8n fails constraint requirements. YOLOv5n selected as final model.'
    else:
        decision = 'YOLOv5n'
        reason = (
            f'YOLOv8n does not outperform YOLOv5n (delta: {delta_map50:+.4f} mAP@0.5). '
            f'YOLOv5n selected as final model per baseline-first protocol.'
        )
    
    print(f'FINAL MODEL: {decision}')
    print()
    print(f'Justification (for Proposal Bab 2):')
    print(f'{reason}')
    print()
    print('Copy this justification into Proposal Bab 2 (Metodologi AI > Pemilihan Arsitektur).')

ARCHITECTURE SELECTION DECISION
YOLOv8n accuracy improvement: -0.1641 mAP@0.5
YOLOv8n speed delta:          +12.4 ms (mean)
YOLOv8n passes all constraints: True

FINAL MODEL: YOLOv5n

Justification (for Proposal Bab 2):
YOLOv8n does not outperform YOLOv5n (delta: -0.1641 mAP@0.5). YOLOv5n selected as final model per baseline-first protocol.

Copy this justification into Proposal Bab 2 (Metodologi AI > Pemilihan Arsitektur).


## 7. Save Comparison Results

In [ ]:
from datetime import datetime

report_path = REPO_ROOT / 'docs' / 'architecture_comparison.txt'
report_path.parent.mkdir(exist_ok=True)

with open(report_path, 'w') as f:
    f.write(f'Architecture Comparison Report\n')
    f.write(f'Generated: {datetime.now().isoformat()}\n')
    f.write(f'Dataset: VisDrone-DET 2019 (pedestrian class, imgsz=416)\n')
    f.write('='*65 + '\n\n')
    
    f.write(f'{"Metric":<28} {"YOLOv5n":>15} {"YOLOv8n":>15}\n')
    f.write('-'*65 + '\n')
    for label, v5_val, v8_val, fmt in rows:
        v5_str = fmt.format(v5_val) if v5_val is not None else 'pending'
        v8_str = fmt.format(v8_val) if v8_val is not None else 'pending'
        f.write(f'{label:<28} {v5_str:>15} {v8_str:>15}\n')
    
    if all_filled:
        f.write('\n' + '='*65 + '\n')
        f.write(f'DECISION: {decision}\n')
        f.write(f'Reason: {reason}\n')

print(f'Report saved: {report_path}')

Report saved: c:\Users\MASTER CORE\RescueVision\docs\architecture_comparison.txt


: 